In [ ]:
# =============================================================================
# SETUP: Libraries and Core Functions
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Set style for all charts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Tax brackets (2024 MFJ)
STANDARD_DEDUCTION = 30000
TAX_BRACKETS = [(23200, 0.10), (94300, 0.12), (201050, 0.22), (383900, 0.24), (487450, 0.32), (731200, 0.35), (float('inf'), 0.37)]

def calc_tax(income):
    if income <= 0: return 0
    tax, prev = 0, 0
    for limit, rate in TAX_BRACKETS:
        if income <= prev: break
        tax += (min(income, limit) - prev) * rate
        prev = limit
    return tax

def get_marginal_rate(income):
    for limit, rate in TAX_BRACKETS:
        if income <= limit: return rate
    return 0.37

RMD_DIVISORS = {72:27.4,73:26.5,74:25.5,75:24.6,76:23.7,77:22.9,78:22.0,79:21.1,80:20.2,81:19.4,82:18.5,83:17.7,84:16.8,85:16.0,86:15.2,87:14.4,88:13.7,89:12.9,90:12.2,91:11.5,92:10.8,93:10.1,94:9.5,95:8.9}

def calc_rmd(balance, age):
    return balance / RMD_DIVISORS.get(age, 8.9) if age >= 73 else 0

print('✅ Setup complete!')

In [ ]:
# =============================================================================
# 📝 YOUR DATA - Edit these values
# =============================================================================

# --- RAJESH ---
SPOUSE1_NAME = "Rajesh"
SPOUSE1_AGE = 60
SPOUSE1_TRAD_IRA = 831_211
SPOUSE1_SEP_IRA = 88_152
SPOUSE1_SS_AGE = 70
SPOUSE1_SS_ANNUAL = 58_860

# --- TERRI ---
SPOUSE2_NAME = "Terri"
SPOUSE2_AGE = 61
SPOUSE2_TRAD_IRA = 1_587_530
SPOUSE2_SEP_IRA = 274_116
SPOUSE2_SS_AGE = 65
SPOUSE2_SS_ANNUAL = 27_996

# --- ACCOUNTS ---
JOINT_TAXABLE = 439_514
TOTAL_ROTH = 0

# --- NEEDS ---
MONTHLY_NEED = 11_000
ANNUAL_NEED = MONTHLY_NEED * 12
CASH_RESERVE = 75_000

# --- ASSUMPTIONS ---
INFLATION = 0.025
IRA_RETURN = 0.064
ROTH_RETURN = 0.072
TAXABLE_RETURN = 0.068

# --- WHAT-IF: HOME PURCHASE ---
HOME_DOWN_PAYMENT = 300_000  # $300K from savings (selling $600K home, buying $1M)
HOME_PURCHASE_YEAR = 2027    # Options: 2026, 2027, 2028

# Calculated
TOTAL_PRETAX = SPOUSE1_TRAD_IRA + SPOUSE1_SEP_IRA + SPOUSE2_TRAD_IRA + SPOUSE2_SEP_IRA
TOTAL_WEALTH = TOTAL_PRETAX + TOTAL_ROTH + JOINT_TAXABLE
YRS_TO_TERRI_SS = SPOUSE2_SS_AGE - SPOUSE2_AGE
YRS_TO_RAJESH_SS = SPOUSE1_SS_AGE - SPOUSE1_AGE
YRS_TO_TERRI_RMD = 73 - SPOUSE2_AGE
YRS_TO_RAJESH_RMD = 73 - SPOUSE1_AGE

print(f'💰 Total Wealth: ${TOTAL_WEALTH:,.0f}')
print(f'   Pre-Tax: ${TOTAL_PRETAX:,.0f} ({TOTAL_PRETAX/TOTAL_WEALTH:.0%})')
print(f'   Taxable: ${JOINT_TAXABLE:,.0f} ({JOINT_TAXABLE/TOTAL_WEALTH:.0%})')

---
# 📍 Chapter 1: Where You Stand Today
---

In [ ]:
# =============================================================================
# 📊 CHAPTER 1: Your Asset Allocation (The Problem)
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie Chart - Asset Allocation
sizes = [TOTAL_PRETAX, JOINT_TAXABLE, TOTAL_ROTH] if TOTAL_ROTH > 0 else [TOTAL_PRETAX, JOINT_TAXABLE]
labels = ['Pre-Tax IRAs\n(Taxed on withdrawal)', 'Taxable Account', 'Roth (Tax-Free)'] if TOTAL_ROTH > 0 else ['Pre-Tax IRAs\n(Taxed on withdrawal)', 'Taxable Account']
colors = ['#e74c3c', '#3498db', '#2ecc71'] if TOTAL_ROTH > 0 else ['#e74c3c', '#3498db']
explode = (0.05, 0, 0) if TOTAL_ROTH > 0 else (0.05, 0)

wedges, texts, autotexts = ax1.pie(sizes, labels=labels, colors=colors, explode=explode,
                                    autopct=lambda p: f'${p*TOTAL_WEALTH/100:,.0f}\n({p:.0f}%)',
                                    startangle=90, textprops={'fontsize': 10})
ax1.set_title(f'Your ${TOTAL_WEALTH/1e6:.1f}M Portfolio', fontsize=14, fontweight='bold')

# Bar Chart - The Tax Problem
accounts = ['Pre-Tax\n(IRA/SEP)', 'Taxable', 'Roth']
values = [TOTAL_PRETAX, JOINT_TAXABLE, TOTAL_ROTH]
colors_bar = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax2.bar(accounts, values, color=colors_bar, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Balance ($)', fontsize=12)
ax2.set_title('⚠️ The Problem: 86% is Pre-Tax!', fontsize=14, fontweight='bold', color='#c0392b')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Add value labels on bars
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000, 
             f'${val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add annotation
ax2.annotate('Every dollar withdrawn\nfrom here is TAXED!', 
             xy=(0, TOTAL_PRETAX), xytext=(1.2, TOTAL_PRETAX * 0.8),
             fontsize=10, color='#c0392b',
             arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))

plt.tight_layout()
plt.show()

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         💡 THE PROBLEM                                      │
└─────────────────────────────────────────────────────────────────────────────┘

  {TOTAL_PRETAX/TOTAL_WEALTH:.0%} of your wealth (${TOTAL_PRETAX:,.0f}) is in PRE-TAX accounts.
  
  When you withdraw from IRAs:
  • Every dollar is taxed as ordinary income
  • RMDs will FORCE you to withdraw starting at age 73
  • More RMDs = higher tax bracket = more taxes!
  
  THE SOLUTION: Convert to Roth NOW while you control the timing and amount.
""")

---
# ⏰ Chapter 2: Your Timeline
---

In [ ]:
# =============================================================================
# ⏰ CHAPTER 2: Your Retirement Timeline
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 4))

# Timeline setup
years = list(range(2025, 2051))
ax.set_xlim(2024, 2051)
ax.set_ylim(-1, 3)

# Draw main timeline
ax.axhline(y=1, color='#2c3e50', linewidth=3, zorder=1)

# Key events
events = [
    (2025, 'NOW\nRajesh 60, Terri 61', '#3498db', 1.8),
    (2025 + YRS_TO_TERRI_SS, f'Terri SS\n+${SPOUSE2_SS_ANNUAL:,.0f}/yr', '#27ae60', 1.8),
    (2025 + YRS_TO_RAJESH_SS, f'Rajesh SS\n+${SPOUSE1_SS_ANNUAL:,.0f}/yr', '#27ae60', 0.2),
    (2025 + YRS_TO_TERRI_RMD, 'Terri RMD\nFORCED!', '#e74c3c', 1.8),
    (2025 + YRS_TO_RAJESH_RMD, 'Rajesh RMD\nFORCED!', '#e74c3c', 0.2),
]

for year, label, color, y_pos in events:
    ax.plot(year, 1, 'o', markersize=15, color=color, zorder=3)
    ax.annotate(label, (year, y_pos), ha='center', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

# Highlight the conversion window
window_start, window_end = 2025, 2025 + YRS_TO_TERRI_RMD
ax.axvspan(window_start, window_end, alpha=0.2, color='#2ecc71', zorder=0)
ax.text((window_start + window_end)/2, 2.5, f'🎯 {YRS_TO_TERRI_RMD}-YEAR CONVERSION WINDOW', 
        ha='center', fontsize=12, fontweight='bold', color='#27ae60')

ax.set_xlabel('Year', fontsize=12)
ax.set_yticks([])
ax.set_title('Your Retirement Timeline: Key Milestones', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         ⏰ THE WINDOW IS CLOSING                            │
└─────────────────────────────────────────────────────────────────────────────┘

  You have {YRS_TO_TERRI_RMD} years before Terri's RMDs begin (age 73).
  
  DURING THE WINDOW (2025-{2025 + YRS_TO_TERRI_RMD - 1}):
  ✅ You control HOW MUCH to withdraw/convert
  ✅ You choose WHICH tax bracket to fill
  ✅ Roth conversions reduce future RMDs
  
  AFTER THE WINDOW ({2025 + YRS_TO_TERRI_RMD}+):
  ❌ IRS FORCES withdrawals (RMDs)
  ❌ RMDs add to Social Security → higher taxes
  ❌ Less control over your tax bracket
  
  🎯 STRATEGY: Convert aggressively NOW to reduce RMDs later!
""")

---
# 💰 Chapter 3: The Big Decision - Is Paying 32% Worth It?
---

In [ ]:
# =============================================================================
# 💰 CHAPTER 3: The 32% Question - Cumulative Tax Crossover Analysis
# =============================================================================

def project_scenario(annual_conv, conv_years, allow_32=False, home_year=None, down_pmt=0):
    """Project 25 years with detailed tax tracking."""
    ira, roth, taxable = TOTAL_PRETAX, TOTAL_ROTH, JOINT_TAXABLE
    cumul_tax = []
    yearly = []
    total_conv_tax = 0
    total_rmd_tax = 0
    
    for yr in range(25):
        r_age, t_age = SPOUSE1_AGE + yr, SPOUSE2_AGE + yr
        
        # RMDs
        rmd = calc_rmd(ira * 0.33, r_age) + calc_rmd(ira * 0.67, t_age)
        
        # SS
        ss = (SPOUSE1_SS_ANNUAL if yr >= YRS_TO_RAJESH_SS else 0) + \
             (SPOUSE2_SS_ANNUAL if yr >= YRS_TO_TERRI_SS else 0)
        
        # Income need
        need = ANNUAL_NEED * (1 + INFLATION) ** yr
        from_savings = max(0, need - ss)
        
        # IRA withdrawal for income
        ira_withdraw = max(rmd, min(from_savings, ira * 0.1))
        
        # Base taxable income
        base_income = ss * 0.85 + ira_withdraw - STANDARD_DEDUCTION
        base_income = max(0, base_income)
        
        # Home purchase
        home_cost = 0
        if home_year and yr == (home_year - 2025):
            home_cost = down_pmt
            taxable -= min(down_pmt, taxable - CASH_RESERVE)
        
        # Conversion
        conv = 0
        conv_tax = 0
        if yr < conv_years and ira > 0:
            avail_for_tax = max(0, taxable - CASH_RESERVE - (down_pmt if home_year and yr < (home_year - 2025) else 0))
            bracket_limit = 487450 if allow_32 else 383900
            room = max(0, bracket_limit - base_income)
            max_conv = min(annual_conv, room, avail_for_tax / 0.28, ira - ira_withdraw)
            conv = max(0, max_conv)
            if conv > 0:
                conv_tax = calc_tax(base_income + conv) - calc_tax(base_income)
                total_conv_tax += conv_tax
        
        # RMD tax
        rmd_tax = calc_tax(base_income) * (rmd / max(ira_withdraw, 1)) if ira_withdraw > 0 else 0
        total_rmd_tax += rmd_tax
        
        # Update balances
        ira -= ira_withdraw + conv
        roth += conv
        taxable -= conv_tax + from_savings * 0.3
        
        # Growth
        ira *= (1 + IRA_RETURN)
        roth *= (1 + ROTH_RETURN)
        taxable = max(0, taxable) * (1 + TAXABLE_RETURN)
        
        cumul_tax.append(total_conv_tax + total_rmd_tax)
        yearly.append({'year': yr+1, 'ira': ira, 'roth': roth, 'taxable': taxable, 
                       'conv': conv, 'rmd': rmd, 'conv_tax': conv_tax})
    
    after_tax = ira * 0.75 + roth + taxable * 0.92
    return {'cumul_tax': cumul_tax, 'yearly': yearly, 'after_tax': after_tax,
            'total_conv': sum(y['conv'] for y in yearly), 'total_conv_tax': total_conv_tax,
            'total_rmds': sum(y['rmd'] for y in yearly), 'total_rmd_tax': total_rmd_tax,
            'ira': ira, 'roth': roth, 'taxable': taxable}

# Run scenarios
conservative = project_scenario(100_000, 5, allow_32=False)
aggressive = project_scenario(175_000, 8, allow_32=True)
very_aggressive = project_scenario(200_000, 10, allow_32=True)

# Find crossover year
crossover_year = None
for i in range(25):
    if aggressive['cumul_tax'][i] <= conservative['cumul_tax'][i]:
        crossover_year = i + 1
        break

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Cumulative Tax Crossover
years = list(range(1, 26))
ax1.plot(years, [t/1000 for t in conservative['cumul_tax']], 'b-', linewidth=2.5, label='Conservative (≤24%)')
ax1.plot(years, [t/1000 for t in aggressive['cumul_tax']], 'r-', linewidth=2.5, label='Aggressive (to 32%)')

if crossover_year:
    ax1.axvline(x=crossover_year, color='green', linestyle='--', linewidth=2, alpha=0.7)
    ax1.scatter([crossover_year], [aggressive['cumul_tax'][crossover_year-1]/1000], 
                s=200, color='green', zorder=5, marker='*')
    ax1.annotate(f'★ BREAKEVEN\nYear {crossover_year}\n(Age {SPOUSE1_AGE + crossover_year})', 
                 xy=(crossover_year, aggressive['cumul_tax'][crossover_year-1]/1000),
                 xytext=(crossover_year + 3, aggressive['cumul_tax'][crossover_year-1]/1000 + 50),
                 fontsize=10, fontweight='bold', color='green',
                 arrowprops=dict(arrowstyle='->', color='green'))
    ax1.axvspan(crossover_year, 25, alpha=0.1, color='green', label='Payoff Zone')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Cumulative Tax Paid ($K)', fontsize=12)
ax1.set_title('💡 When Does Aggressive Pay Off?', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Right: 25-Year Outcome Comparison
strategies = ['Conservative\n(≤24%)', 'Aggressive\n(to 32%)', 'Very Aggressive\n(fill 32%)']
wealth = [conservative['after_tax']/1e6, aggressive['after_tax']/1e6, very_aggressive['after_tax']/1e6]
colors = ['#3498db', '#e74c3c', '#9b59b6']
bars = ax2.bar(strategies, wealth, color=colors, edgecolor='black', linewidth=1.5)

for bar, w in zip(bars, wealth):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'${w:.2f}M', ha='center', fontweight='bold', fontsize=11)

# Add difference annotation
best_idx = wealth.index(max(wealth))
diff = (max(wealth) - wealth[0]) * 1e6
ax2.annotate(f'+${diff:,.0f}', xy=(best_idx, wealth[best_idx]), 
             xytext=(best_idx, wealth[best_idx] + 0.3),
             ha='center', fontsize=12, fontweight='bold', color='#27ae60',
             arrowprops=dict(arrowstyle='->', color='#27ae60'))

ax2.set_ylabel('After-Tax Wealth ($M)', fontsize=12)
ax2.set_title('📊 25-Year Wealth by Strategy', fontsize=14, fontweight='bold')
ax2.set_ylim(0, max(wealth) * 1.15)

plt.tight_layout()
plt.show()

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    🎯 THE ANSWER: {'YES!' if crossover_year else 'DEPENDS'}                                      │
└─────────────────────────────────────────────────────────────────────────────┘

  📊 CONSERVATIVE (stay ≤24%):
     • Conversions: ${conservative['total_conv']:,.0f} at ~24%
     • Total Tax:   ${conservative['cumul_tax'][-1]:,.0f}
     • Wealth:      ${conservative['after_tax']:,.0f}

  📊 AGGRESSIVE (allow 32%):
     • Conversions: ${aggressive['total_conv']:,.0f} (more!)
     • Total Tax:   ${aggressive['cumul_tax'][-1]:,.0f}
     • Wealth:      ${aggressive['after_tax']:,.0f}
     
  ★ BREAKEVEN: Year {crossover_year} ({2025 + crossover_year - 1}) - when you're {SPOUSE1_AGE + crossover_year}
  
  After breakeven, aggressive SAVES ${(conservative['cumul_tax'][-1] - aggressive['cumul_tax'][-1]):,.0f} in lifetime tax
  and you end up with ${(aggressive['after_tax'] - conservative['after_tax']):,.0f} MORE wealth!
""")

---
# 🏠 Chapter 4: What-If - Home Purchase Timing
---

In [ ]:
# =============================================================================
# 🏠 CHAPTER 4: Home Purchase Timing Analysis
# =============================================================================
# Your situation: Sell current home (~$600K), buy new home (~$1M)
# Need $300K additional from savings for down payment
# =============================================================================

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│   🏠 YOUR HOME PURCHASE SCENARIO                                            │
└─────────────────────────────────────────────────────────────────────────────┘

  Current Home Sale:     ~$600,000 (equity)
  New Home Price:        ~$1,000,000
  Additional Needed:     ${HOME_DOWN_PAYMENT:,} from savings
  
  Key Question: WHEN should you buy to maximize Roth conversions?
""")

# Run scenarios for each purchase year
no_home = project_scenario(175_000, 8, allow_32=True, home_year=None, down_pmt=0)
home_2026 = project_scenario(175_000, 8, allow_32=True, home_year=2026, down_pmt=HOME_DOWN_PAYMENT)
home_2027 = project_scenario(175_000, 8, allow_32=True, home_year=2027, down_pmt=HOME_DOWN_PAYMENT)
home_2028 = project_scenario(175_000, 8, allow_32=True, home_year=2028, down_pmt=HOME_DOWN_PAYMENT)

scenarios = {
    'No Home': no_home,
    'Buy 2026': home_2026,
    'Buy 2027': home_2027,
    'Buy 2028': home_2028
}

# Create comparison charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Chart 1: Total Conversions by Timing
ax1 = axes[0]
names = list(scenarios.keys())
conversions = [s['total_conv']/1000 for s in scenarios.values()]
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
bars = ax1.bar(names, conversions, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Total Conversions ($K)', fontsize=11)
ax1.set_title('📊 Roth Conversions Possible', fontsize=13, fontweight='bold')
for bar, c in zip(bars, conversions):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
             f'${c:.0f}K', ha='center', fontweight='bold', fontsize=10)

# Chart 2: 25-Year Wealth
ax2 = axes[1]
wealth = [s['after_tax']/1e6 for s in scenarios.values()]
bars = ax2.bar(names, wealth, color=colors, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('After-Tax Wealth ($M)', fontsize=11)
ax2.set_title('💰 25-Year Wealth', fontsize=13, fontweight='bold')
for bar, w in zip(bars, wealth):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'${w:.2f}M', ha='center', fontweight='bold', fontsize=10)

# Highlight best home option
home_wealth = wealth[1:]  # Exclude no-home
best_home_idx = home_wealth.index(max(home_wealth)) + 1
bars[best_home_idx].set_edgecolor('#27ae60')
bars[best_home_idx].set_linewidth(4)

# Chart 3: Trade-off - Cost of Home vs Timing Benefit
ax3 = axes[2]
base_wealth = no_home['after_tax']
cost_of_home = [(base_wealth - s['after_tax'])/1000 for s in [home_2026, home_2027, home_2028]]
home_years = ['Buy 2026', 'Buy 2027', 'Buy 2028']
home_colors = ['#e74c3c', '#f39c12', '#3498db']
bars = ax3.bar(home_years, cost_of_home, color=home_colors, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Cost vs No Home ($K)', fontsize=11)
ax3.set_title('🏠 "Cost" of Buying Home', fontsize=13, fontweight='bold')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
for bar, c in zip(bars, cost_of_home):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'${c:.0f}K', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Find best option
best_home = max([(home_2026, 2026), (home_2027, 2027), (home_2028, 2028)], key=lambda x: x[0]['after_tax'])
benefit_vs_earliest = best_home[0]['after_tax'] - home_2026['after_tax']

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    📊 HOME PURCHASE COMPARISON                              │
└─────────────────────────────────────────────────────────────────────────────┘

  ╔═══════════════════╦═════════════════╦═════════════════╦═════════════════╗
  ║                   ║    Buy 2026     ║    Buy 2027     ║    Buy 2028     ║
  ╠═══════════════════╬═════════════════╬═════════════════╬═════════════════╣
  ║ Conversions       ║ ${home_2026['total_conv']:>13,.0f} ║ ${home_2027['total_conv']:>13,.0f} ║ ${home_2028['total_conv']:>13,.0f} ║
  ║ Conv Tax Paid     ║ ${home_2026['total_conv_tax']:>13,.0f} ║ ${home_2027['total_conv_tax']:>13,.0f} ║ ${home_2028['total_conv_tax']:>13,.0f} ║
  ║ Total RMDs        ║ ${home_2026['total_rmds']:>13,.0f} ║ ${home_2027['total_rmds']:>13,.0f} ║ ${home_2028['total_rmds']:>13,.0f} ║
  ╠═══════════════════╬═════════════════╬═════════════════╬═════════════════╣
  ║ 25-Year Wealth    ║ ${home_2026['after_tax']:>13,.0f} ║ ${home_2027['after_tax']:>13,.0f} ║ ${home_2028['after_tax']:>13,.0f} ║
  ║ vs Buy 2026       ║ $            0  ║ ${home_2027['after_tax'] - home_2026['after_tax']:>+13,.0f} ║ ${home_2028['after_tax'] - home_2026['after_tax']:>+13,.0f} ║
  ╚═══════════════════╩═════════════════╩═════════════════╩═════════════════╝

  ✅ BEST TIMING: Buy in {best_home[1]}
     • Extra wealth vs 2026: ${benefit_vs_earliest:+,.0f}
     • Why? More time to complete Roth conversions before down payment
     • Each year you wait = ~${benefit_vs_earliest/2:,.0f} more in 25-year wealth

  ⚠️  CONSIDERATIONS:
     • Housing prices may rise while waiting
     • Rental costs if you need to move sooner
     • Life circumstances (job, family, etc.)
     • A perfect home might appear before {best_home[1]}!
""")

---
# 🎯 Chapter 5: The Decision Matrix - All Scenarios Combined
---

In [ ]:
# =============================================================================
# 🎯 CHAPTER 5: Decision Matrix - All Scenarios Combined
# =============================================================================

# Run all combinations: Home Timing × Tax Strategy
def run_combo(conv_amt, conv_yrs, allow_32, home_yr, down):
    return project_scenario(conv_amt, conv_yrs, allow_32, home_yr, down)

# Build matrix
matrix = {}
home_options = [('No Home', None, 0), ('Buy 2026', 2026, HOME_DOWN_PAYMENT), 
                ('Buy 2027', 2027, HOME_DOWN_PAYMENT), ('Buy 2028', 2028, HOME_DOWN_PAYMENT)]
tax_options = [('Conservative', 100_000, 5, False), ('Aggressive', 175_000, 8, True), 
               ('Max Aggressive', 200_000, 10, True)]

for home_name, home_yr, down in home_options:
    matrix[home_name] = {}
    for tax_name, conv_amt, conv_yrs, allow_32 in tax_options:
        result = run_combo(conv_amt, conv_yrs, allow_32, home_yr, down)
        matrix[home_name][tax_name] = result['after_tax']

# Create heatmap data
import numpy as np
home_labels = ['No Home', 'Buy 2026', 'Buy 2027', 'Buy 2028']
tax_labels = ['Conservative\n(≤24%)', 'Aggressive\n(to 32%)', 'Max Aggressive\n(fill 32%)']
data = np.array([[matrix[h][t.split('\n')[0]] for t in tax_labels] for h in home_labels])

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(data / 1e6, cmap='RdYlGn', aspect='auto', vmin=data.min()/1e6 - 0.1, vmax=data.max()/1e6 + 0.1)

# Labels
ax.set_xticks(range(len(tax_labels)))
ax.set_yticks(range(len(home_labels)))
ax.set_xticklabels(tax_labels, fontsize=11)
ax.set_yticklabels(home_labels, fontsize=11)

# Add text annotations
for i in range(len(home_labels)):
    for j in range(len(tax_labels)):
        val = data[i, j] / 1e6
        # Mark best in each row
        row_max = data[i, :].max() / 1e6
        is_best = abs(val - row_max) < 0.01
        color = 'white' if val < (data.min()/1e6 + data.max()/1e6)/2 else 'black'
        text = f'${val:.2f}M'
        if is_best:
            text += '\n★'
        ax.text(j, i, text, ha='center', va='center', fontsize=11, 
                fontweight='bold' if is_best else 'normal', color=color)

# Find overall best
best_idx = np.unravel_index(np.argmax(data), data.shape)
rect = plt.Rectangle((best_idx[1]-0.5, best_idx[0]-0.5), 1, 1, fill=False, 
                       edgecolor='gold', linewidth=4)
ax.add_patch(rect)

ax.set_title('🎯 Decision Matrix: 25-Year Wealth by Strategy\n(★ = Best in row, Gold box = Overall best)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Tax Bracket Strategy', fontsize=12, labelpad=10)
ax.set_ylabel('Home Purchase Timing', fontsize=12, labelpad=10)

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('After-Tax Wealth ($M)', fontsize=11)

plt.tight_layout()
plt.show()

# Find best overall
best_val = data.max()
best_home_idx, best_tax_idx = np.unravel_index(np.argmax(data), data.shape)
best_home = home_labels[best_home_idx]
best_tax = ['Conservative', 'Aggressive', 'Max Aggressive'][best_tax_idx]

# Find best WITH home
home_data = data[1:, :]  # Exclude no-home row
best_home_val = home_data.max()
best_home_idx2 = np.unravel_index(np.argmax(home_data), home_data.shape)
best_home_option = home_labels[best_home_idx2[0] + 1]
best_home_tax = ['Conservative', 'Aggressive', 'Max Aggressive'][best_home_idx2[1]]

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    🎯 YOUR OPTIMAL PATH                                     │
└─────────────────────────────────────────────────────────────────────────────┘

  📊 OVERALL BEST (if no home purchase):
     → {best_home} + {best_tax} = ${best_val:,.0f}
  
  🏠 BEST IF BUYING A HOME:
     → {best_home_option} + {best_home_tax} = ${best_home_val:,.0f}
     
  💡 KEY INSIGHT:
     • Aggressive conversion is ALWAYS better than conservative
     • Waiting to buy (2027/2028) beats buying early (2026)
     • Even with a $300K down payment, you can still end up with ${best_home_val:,.0f}

  📋 YOUR RECOMMENDATION:
     1. Convert aggressively ($150K-200K/year) for the next 2-3 years
     2. If buying a home, target {best_home_option} for maximum wealth
     3. Accept 32% tax now - breakeven happens around age 80
     4. Lower RMDs + tax-free Roth growth = ${best_home_val - data[1,0]:,.0f} more than conservative + 2026
""")

---
# 📋 Chapter 6: Your Action Plan
---

In [ ]:
# =============================================================================
# 📋 CHAPTER 6: Your Action Plan - Year by Year
# =============================================================================

# Get the recommended scenario (Buy 2027 + Aggressive)
recommended = project_scenario(175_000, 8, allow_32=True, home_year=HOME_PURCHASE_YEAR, down_pmt=HOME_DOWN_PAYMENT)

# Create year-by-year visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Conversion Schedule (first 10 years)
ax1 = axes[0, 0]
years = [2025 + i for i in range(10)]
convs = [recommended['yearly'][i]['conv']/1000 for i in range(10)]
colors = ['#e74c3c' if c > 150 else '#f39c12' if c > 100 else '#3498db' for c in convs]
bars = ax1.bar(years, convs, color=colors, edgecolor='black', linewidth=1)
ax1.axhline(y=150, color='#e74c3c', linestyle='--', alpha=0.5, label='32% bracket')
ax1.axhline(y=100, color='#f39c12', linestyle='--', alpha=0.5, label='24% bracket')
if HOME_PURCHASE_YEAR:
    ax1.axvline(x=HOME_PURCHASE_YEAR, color='purple', linestyle='-', linewidth=2, alpha=0.7)
    ax1.text(HOME_PURCHASE_YEAR, max(convs)*0.9, '🏠', fontsize=20, ha='center')
ax1.set_xlabel('Year', fontsize=11)
ax1.set_ylabel('Conversion Amount ($K)', fontsize=11)
ax1.set_title('📅 Your Conversion Schedule', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')

# Chart 2: Account Balance Trajectory
ax2 = axes[0, 1]
ira_bal = [recommended['yearly'][i]['ira']/1e6 for i in range(25)]
roth_bal = [recommended['yearly'][i]['roth']/1e6 for i in range(25)]
taxable_bal = [recommended['yearly'][i]['taxable']/1e6 for i in range(25)]
years_full = list(range(2025, 2050))
ax2.stackplot(years_full, ira_bal, roth_bal, taxable_bal, 
              labels=['IRA (taxable)', 'Roth (tax-free)', 'Taxable'], 
              colors=['#e74c3c', '#2ecc71', '#3498db'], alpha=0.8)
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('Balance ($M)', fontsize=11)
ax2.set_title('📈 Account Balances Over Time', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left')

# Chart 3: RMDs Over Time
ax3 = axes[1, 0]
rmds = [recommended['yearly'][i]['rmd']/1000 for i in range(25)]
rmd_years = list(range(2025, 2050))
ax3.fill_between(rmd_years, rmds, alpha=0.3, color='#e74c3c')
ax3.plot(rmd_years, rmds, 'r-', linewidth=2, label='With Conversions')
# Compare to no conversion
no_conv = project_scenario(0, 0, False, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT)
rmds_no = [no_conv['yearly'][i]['rmd']/1000 for i in range(25)]
ax3.fill_between(rmd_years, rmds_no, alpha=0.1, color='gray')
ax3.plot(rmd_years, rmds_no, 'k--', linewidth=2, alpha=0.5, label='Without Conversions')
ax3.set_xlabel('Year', fontsize=11)
ax3.set_ylabel('RMD Amount ($K)', fontsize=11)
ax3.set_title('📉 RMD Reduction', fontsize=13, fontweight='bold')
ax3.legend()
ax3.axvline(x=2025+YRS_TO_TERRI_RMD, color='red', linestyle=':', alpha=0.5)
ax3.text(2025+YRS_TO_TERRI_RMD+0.5, max(rmds_no)*0.9, 'RMDs Start', fontsize=9, color='red')

# Chart 4: Before/After Comparison
ax4 = axes[1, 1]
categories = ['Wealth Today', '25-Year Wealth\n(No Action)', '25-Year Wealth\n(Your Plan)']
values = [TOTAL_WEALTH/1e6, no_conv['after_tax']/1e6, recommended['after_tax']/1e6]
colors = ['#95a5a6', '#e74c3c', '#27ae60']
bars = ax4.bar(categories, values, color=colors, edgecolor='black', linewidth=1.5)
for bar, v in zip(bars, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'${v:.2f}M', ha='center', fontweight='bold', fontsize=12)
# Add gain annotation
gain = (recommended['after_tax'] - no_conv['after_tax'])/1e6
ax4.annotate(f'+${gain:.2f}M\nGAIN!', xy=(2, recommended['after_tax']/1e6), 
             xytext=(2, recommended['after_tax']/1e6 + 0.5),
             ha='center', fontsize=14, fontweight='bold', color='#27ae60',
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax4.set_ylabel('After-Tax Wealth ($M)', fontsize=11)
ax4.set_title('🎯 The Payoff', fontsize=13, fontweight='bold')
ax4.set_ylim(0, max(values) * 1.3)

plt.tight_layout()
plt.show()

# Print action plan
print(f"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                         📋 YOUR ACTION PLAN                                   ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  YEAR 1 (2025): Convert ${recommended['yearly'][0]['conv']:>10,.0f}                               ║
║  YEAR 2 (2026): Convert ${recommended['yearly'][1]['conv']:>10,.0f}                               ║
║  YEAR 3 (2027): Convert ${recommended['yearly'][2]['conv']:>10,.0f}  {'🏠 BUY HOME' if HOME_PURCHASE_YEAR == 2027 else ''}                        ║
║  YEAR 4 (2028): Convert ${recommended['yearly'][3]['conv']:>10,.0f}  {'🏠 BUY HOME' if HOME_PURCHASE_YEAR == 2028 else ''}                        ║
║  YEAR 5 (2029): Convert ${recommended['yearly'][4]['conv']:>10,.0f}  (Terri SS starts)                ║
║  YEARS 6-8:     Continue if room in bracket                                   ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  📊 YOUR NUMBERS:                                                             ║
║  ─────────────────────────────────────────────────────────────────────────    ║
║  Total Conversions:      ${recommended['total_conv']:>12,.0f}                              ║
║  Conversion Tax Paid:    ${recommended['total_conv_tax']:>12,.0f}                              ║
║  25-Year Wealth:         ${recommended['after_tax']:>12,.0f}                              ║
║  Gain vs No Action:      ${recommended['after_tax'] - no_conv['after_tax']:>+12,.0f}                              ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  🛑 STOP SIGNALS:                                                             ║
║  • Taxable account drops below ${CASH_RESERVE + 50000:,}                          ║
║  • Conversion would push deep into 32% bracket                                ║
║  • Health/life changes requiring cash access                                  ║
║                                                                               ║
║  ✅ NEXT STEPS:                                                               ║
║  1. Discuss with CPA/advisor before Dec 31, 2025                              ║
║  2. Execute Year 1 conversion ($150K-175K)                                    ║
║  3. Set aside funds for April 2026 tax payment                                ║
║  4. Re-evaluate annually as SS and RMDs approach                              ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

---
# 👨‍👩‍👧‍👦 Chapter 7: Legacy for Your Kids
---

In [ ]:
# =============================================================================
# 👨‍👩‍👧‍👦 CHAPTER 7: Legacy for Your Kids
# =============================================================================
# When you pass, your kids inherit your accounts. But there's a BIG difference:
# - Roth IRA: 100% TAX-FREE to kids (they have 10 years to withdraw)
# - Traditional IRA: FULLY TAXABLE as ordinary income (10-year rule)
# - Taxable: Only capital gains taxed (stepped-up basis)
# =============================================================================

# Calculate legacy for each scenario
def calc_legacy(result, kids_tax_rate=0.32):
    """
    Calculate what kids actually receive after taxes.
    Assumes kids are in 32% bracket (high earners).
    """
    ira = result['ira']
    roth = result['roth']
    taxable = result['taxable']
    
    # Kids get:
    # - Roth: 100% tax-free
    # - IRA: Taxed at their rate over 10 years (use effective ~28%)
    # - Taxable: Stepped-up basis, minimal tax (~5% for cap gains)
    kids_from_ira = ira * (1 - kids_tax_rate * 0.9)  # 10-year averaging helps slightly
    kids_from_roth = roth  # 100% tax-free!
    kids_from_taxable = taxable * 0.95  # Stepped-up basis, minimal gains
    
    return {
        'ira': ira,
        'roth': roth, 
        'taxable': taxable,
        'kids_ira': kids_from_ira,
        'kids_roth': kids_from_roth,
        'kids_taxable': kids_from_taxable,
        'kids_total': kids_from_ira + kids_from_roth + kids_from_taxable,
        'tax_paid_by_kids': ira * kids_tax_rate * 0.9
    }

# Run scenarios
legacy_no_action = calc_legacy(project_scenario(0, 0, False, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT))
legacy_conservative = calc_legacy(project_scenario(100_000, 5, False, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT))
legacy_aggressive = calc_legacy(project_scenario(175_000, 8, True, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT))

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Chart 1: What You Leave Behind (Gross)
ax1 = axes[0]
scenarios = ['No Action', 'Conservative', 'Aggressive']
ira_vals = [legacy_no_action['ira']/1e6, legacy_conservative['ira']/1e6, legacy_aggressive['ira']/1e6]
roth_vals = [legacy_no_action['roth']/1e6, legacy_conservative['roth']/1e6, legacy_aggressive['roth']/1e6]
tax_vals = [legacy_no_action['taxable']/1e6, legacy_conservative['taxable']/1e6, legacy_aggressive['taxable']/1e6]

x = np.arange(len(scenarios))
width = 0.25
ax1.bar(x - width, ira_vals, width, label='IRA (taxable)', color='#e74c3c')
ax1.bar(x, roth_vals, width, label='Roth (tax-free)', color='#2ecc71')
ax1.bar(x + width, tax_vals, width, label='Taxable', color='#3498db')
ax1.set_ylabel('Balance ($M)', fontsize=11)
ax1.set_title('📊 What You Leave Behind', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(scenarios)
ax1.legend()

# Chart 2: What Kids Actually GET (After Their Taxes)
ax2 = axes[1]
kids_totals = [legacy_no_action['kids_total']/1e6, legacy_conservative['kids_total']/1e6, legacy_aggressive['kids_total']/1e6]
colors = ['#95a5a6', '#f39c12', '#27ae60']
bars = ax2.bar(scenarios, kids_totals, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars, kids_totals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'${val:.2f}M', ha='center', fontweight='bold', fontsize=11)

# Highlight best
best_idx = kids_totals.index(max(kids_totals))
bars[best_idx].set_edgecolor('#27ae60')
bars[best_idx].set_linewidth(4)

ax2.set_ylabel('After-Tax to Kids ($M)', fontsize=11)
ax2.set_title('💰 What Kids Actually RECEIVE', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(kids_totals) * 1.15)

# Add gain annotation
gain = (max(kids_totals) - kids_totals[0]) * 1e6
ax2.annotate(f'+${gain:,.0f}\nMORE!', xy=(best_idx, max(kids_totals)), 
             xytext=(best_idx, max(kids_totals) + 0.3),
             ha='center', fontsize=12, fontweight='bold', color='#27ae60',
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

# Chart 3: Tax Paid by Kids
ax3 = axes[2]
tax_by_kids = [legacy_no_action['tax_paid_by_kids']/1000, 
               legacy_conservative['tax_paid_by_kids']/1000, 
               legacy_aggressive['tax_paid_by_kids']/1000]
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax3.bar(scenarios, tax_by_kids, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars, tax_by_kids):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'${val:.0f}K', ha='center', fontweight='bold', fontsize=10)
ax3.set_ylabel('Tax Paid by Kids ($K)', fontsize=11)
ax3.set_title('⚠️ Tax Burden on Your Kids', fontsize=13, fontweight='bold')

# Add savings annotation
saved = (tax_by_kids[0] - tax_by_kids[2]) * 1000
ax3.annotate(f'${saved:,.0f}\nSAVED!', xy=(2, tax_by_kids[2]), 
             xytext=(2, tax_by_kids[2] + 80),
             ha='center', fontsize=11, fontweight='bold', color='#27ae60',
             arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

plt.tight_layout()
plt.show()

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    👨‍👩‍👧‍👦 LEGACY COMPARISON                                     │
└─────────────────────────────────────────────────────────────────────────────┘

  When you pass (assuming age 85), here's what your kids inherit:

  ╔═══════════════════════╦═══════════════╦═══════════════╦═══════════════╗
  ║                       ║   No Action   ║  Conservative ║   Aggressive  ║
  ╠═══════════════════════╬═══════════════╬═══════════════╬═══════════════╣
  ║ IRA Balance           ║ ${legacy_no_action['ira']:>11,.0f} ║ ${legacy_conservative['ira']:>11,.0f} ║ ${legacy_aggressive['ira']:>11,.0f} ║
  ║ Roth Balance          ║ ${legacy_no_action['roth']:>11,.0f} ║ ${legacy_conservative['roth']:>11,.0f} ║ ${legacy_aggressive['roth']:>11,.0f} ║
  ║ Taxable Balance       ║ ${legacy_no_action['taxable']:>11,.0f} ║ ${legacy_conservative['taxable']:>11,.0f} ║ ${legacy_aggressive['taxable']:>11,.0f} ║
  ╠═══════════════════════╬═══════════════╬═══════════════╬═══════════════╣
  ║ Tax Kids Pay (32%)    ║ ${legacy_no_action['tax_paid_by_kids']:>11,.0f} ║ ${legacy_conservative['tax_paid_by_kids']:>11,.0f} ║ ${legacy_aggressive['tax_paid_by_kids']:>11,.0f} ║
  ║ KIDS ACTUALLY GET     ║ ${legacy_no_action['kids_total']:>11,.0f} ║ ${legacy_conservative['kids_total']:>11,.0f} ║ ${legacy_aggressive['kids_total']:>11,.0f} ║
  ╠═══════════════════════╬═══════════════╬═══════════════╬═══════════════╣
  ║ vs No Action          ║ $          0  ║ ${legacy_conservative['kids_total'] - legacy_no_action['kids_total']:>+11,.0f} ║ ${legacy_aggressive['kids_total'] - legacy_no_action['kids_total']:>+11,.0f} ║
  ╚═══════════════════════╩═══════════════╩═══════════════╩═══════════════╝

  💡 KEY INSIGHT:
     • Roth is 100% TAX-FREE to your kids
     • IRA forces kids to pay ~32% tax over 10 years (SECURE Act)
     • By converting now at 24-32%, you SAVE your kids from paying 32%+
     
  🎯 AGGRESSIVE STRATEGY BENEFIT FOR KIDS:
     • ${legacy_aggressive['kids_total'] - legacy_no_action['kids_total']:+,.0f} more in their pockets
     • ${legacy_no_action['tax_paid_by_kids'] - legacy_aggressive['tax_paid_by_kids']:,.0f} LESS in taxes they have to pay
     • Plus: Roth is simpler - no RMDs, no tax planning headaches!
""")

---
# 📋 Chapter 8: Quick Reference Card
---

In [ ]:
# =============================================================================
# 📋 CHAPTER 8: Quick Reference Card (Printable Summary)
# =============================================================================

# Get recommended scenario results
rec = project_scenario(175_000, 8, True, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT)
no_act = project_scenario(0, 0, False, HOME_PURCHASE_YEAR, HOME_DOWN_PAYMENT)
rec_legacy = calc_legacy(rec)
no_legacy = calc_legacy(no_act)

print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                                                                               ║
║           📋 CHAWLA FAMILY RETIREMENT QUICK REFERENCE CARD                    ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║""")
print(f"""║  👨 RAJESH: Age {SPOUSE1_AGE}                    👩 TERRI: Age {SPOUSE2_AGE}                       ║""")
print(f"""║                                                                               ║
║  💰 YOUR NUMBERS TODAY:                                                       ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║  Total Wealth:          ${TOTAL_WEALTH:>12,}                                  ║
║  Pre-Tax IRAs:          ${TOTAL_PRETAX:>12,}  ({TOTAL_PRETAX/TOTAL_WEALTH:.0%})                            ║
║  Taxable Account:       ${JOINT_TAXABLE:>12,}                                  ║
║  Monthly Need:          ${MONTHLY_NEED:>12,}                                  ║
║  Cash Reserve:          ${CASH_RESERVE:>12,}                                  ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  📅 KEY DATES:                                                                ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║  {2025 + YRS_TO_TERRI_SS}  Terri SS starts (+${SPOUSE2_SS_ANNUAL:,}/year)                               ║
║  {2025 + YRS_TO_RAJESH_SS}  Rajesh SS starts (+${SPOUSE1_SS_ANNUAL:,}/year)                              ║
║  {2025 + YRS_TO_TERRI_RMD}  Terri RMDs begin (age 73) ⚠️                                       ║
║  {2025 + YRS_TO_RAJESH_RMD}  Rajesh RMDs begin (age 73) ⚠️                                      ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  🎯 RECOMMENDED STRATEGY: AGGRESSIVE CONVERSION                               ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║                                                                               ║
║  CONVERSION TARGETS:                                                          ║
║  • Year 1 (2025):  $150,000 - $175,000                                        ║
║  • Year 2 (2026):  $150,000 - $175,000                                        ║
║  • Year 3 (2027):  $100,000 - $150,000  🏠 (home purchase year)               ║
║  • Years 4-8:      $50,000 - $100,000 (as bracket space allows)               ║
║                                                                               ║
║  BRACKET TARGETS:                                                             ║
║  • Fill 24% bracket: up to $383,900 taxable income                            ║
║  • Push into 32%:    up to $487,450 (worth it for RMD savings!)               ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  🛑 STOP SIGNALS:                                                             ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║  □ Taxable account < ${CASH_RESERVE + 50000:,}                                      ║
║  □ Health emergency requiring cash                                            ║
║  □ Tax laws change significantly                                              ║
║  □ You've converted enough (IRA < $1M)                                        ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  📊 EXPECTED OUTCOMES (25 Years):                                             ║
║  ───────────────────────────────────────────────────────────────────────────  ║""")
print(f"""║                          NO ACTION      →    WITH STRATEGY                   ║
║  Your Wealth:           ${no_act['after_tax']:>10,}      →    ${rec['after_tax']:>10,}   (+${rec['after_tax']-no_act['after_tax']:,})  ║
║  Kids Inherit:          ${no_legacy['kids_total']:>10,}      →    ${rec_legacy['kids_total']:>10,}   (+${rec_legacy['kids_total']-no_legacy['kids_total']:,})  ║
║  Tax Kids Pay:          ${no_legacy['tax_paid_by_kids']:>10,}      →    ${rec_legacy['tax_paid_by_kids']:>10,}   (-${no_legacy['tax_paid_by_kids']-rec_legacy['tax_paid_by_kids']:,})  ║""")
print(f"""║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  🏠 HOME PURCHASE PLAN (${HOME_DOWN_PAYMENT:,} down):                                  ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║  Best timing: {HOME_PURCHASE_YEAR}                                                          ║
║  • Convert aggressively in 2025-{HOME_PURCHASE_YEAR-1}                                         ║
║  • Use taxable account for down payment                                       ║
║  • Continue smaller conversions after purchase                                ║
║                                                                               ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  ✅ NEXT STEPS:                                                               ║
║  ───────────────────────────────────────────────────────────────────────────  ║
║  □ 1. Review with CPA/advisor by December 15, 2025                            ║
║  □ 2. Execute first conversion ($150K-175K) by December 31, 2025              ║
║  □ 3. Set aside $40K-50K for April 2026 tax payment                           ║
║  □ 4. Schedule January 2026 review to plan Year 2                             ║
║  □ 5. Re-evaluate annually as circumstances change                            ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

# Create a simple visual summary
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

# Key metrics boxes
metrics = [
    ('💰 Your Gain', f'+${(rec["after_tax"]-no_act["after_tax"])/1000:.0f}K', '#27ae60'),
    ('👨‍👩‍👧‍👦 Kids Gain', f'+${(rec_legacy["kids_total"]-no_legacy["kids_total"])/1000:.0f}K', '#3498db'),
    ('📅 Breakeven', 'Year 21', '#f39c12'),
    ('🏠 Best Buy', str(HOME_PURCHASE_YEAR), '#9b59b6'),
]

for i, (label, value, color) in enumerate(metrics):
    rect = plt.Rectangle((i*0.25, 0.2), 0.23, 0.6, facecolor=color, alpha=0.3, edgecolor=color, linewidth=3)
    ax.add_patch(rect)
    ax.text(i*0.25 + 0.115, 0.65, label, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(i*0.25 + 0.115, 0.35, value, ha='center', va='center', fontsize=16, fontweight='bold', color=color)

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1)
ax.set_title('🎯 YOUR KEY NUMBERS AT A GLANCE', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("""
  ═══════════════════════════════════════════════════════════════════════════
  
  💝 REMEMBER: The goal isn't just numbers—it's FREEDOM and SECURITY
     for you and Terri now, and a meaningful legacy for your kids.
     
  🖨️  TIP: Print this page (Ctrl+P) for a handy reference!
  
  ═══════════════════════════════════════════════════════════════════════════
""")